# ollada — verify sdar:4b on a T4 (notebook v0.1.1)

**Before you start:** Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all.

This runs ollada's real code (catalog recipe → family adapter → ModelManager) against
SDAR-4B-Chat (bf16 weights, fp16 load). Success = the last cell prints a VERIFICATION RECORD.
If anything crashes, the failing cell *is* the bug report — leave it and tell Muse.
The 5th prompt is an instruction-following probe on purpose: IFEval is SDAR's weak spot in its paper (56.6), so we test it directly.


In [ ]:
# 1 — install (SDAR needs transformers>=4.57 (its modeling imports TransformersKwargs, added in 4.57) — its README's >=4.52.4 claim is stale; the adapter repairs its flash_attn requirement, missing HF file, and BlockMask-only attention at load)
import subprocess, sys
pkgs = [
    "transformers==4.57.0", "torch", "accelerate", "pyyaml",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                "git+https://github.com/OpenSourceRulesz/ollada@main"], check=True)
print("installed")

In [ ]:
# 2 — version stamp + GPU check (acceptance gate)
import torch, ollada
NOTEBOOK_VERSION = "0.1.1"  # also used by the record cell — keep it a variable, not a literal (dream v0.1.0 bug)
print("notebook version:", NOTEBOOK_VERSION)
print("ollada version:", ollada.__version__)
assert torch.cuda.is_available(), "no CUDA — set the runtime to T4 GPU first"
name = torch.cuda.get_device_name(0)
total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {name}  VRAM: {total:.1f} GB")

In [ ]:
# 3 — load sdar:4b through ollada itself (timed)
import time, torch
from ollada.runtime.manager import ModelManager

# Rerun hygiene: if this kernel already loaded a model in a previous run,
# its variables are still bound and the old weights still occupy VRAM —
# loading a second copy OOMs a 16GB card. Drop stale references first.
import gc
for _stale in ("adapter", "manager"):
    globals().pop(_stale, None)
gc.collect()
torch.cuda.empty_cache()

torch.cuda.reset_peak_memory_stats()
t0 = time.time()
manager = ModelManager()
adapter = manager.ensure_loaded("sdar:4b")
load_s = time.time() - t0
peak = torch.cuda.max_memory_allocated() / 1e9
print(f"loaded sdar:4b in {load_s:.1f}s — peak VRAM {peak:.2f} GB")
print("ready on cuda")

In [ ]:
# 4 — fixed prompt set (same 4 as dream:7b for comparability, + an instruction-following probe)
prompts = [
    "In one or two sentences, what is a diffusion language model?",
    "A train travels 60 km in 45 minutes. What is its average speed in km/h? Show the steps.",
    "Write a Python function that returns the nth Fibonacci number, with a docstring.",
    "Give me three tips for staying focused while working from home, as a short list.",
    "Reply with exactly one word: what is the capital of France?",
]
results = []
for p in prompts:
    r = manager.chat("sdar:4b", [{"role": "user", "content": p}])
    tps = r.tokens / r.seconds if r.seconds else 0
    results.append({"prompt": p[:40], "tokens": r.tokens, "seconds": round(r.seconds, 2), "tok_per_s": round(tps, 1)})
    print("=" * 70)
    print("PROMPT:", p)
    print(f"[{r.tokens} tokens in {r.seconds:.1f}s = {tps:.1f} tok/s]")
    print(r.text)

In [ ]:
# 5 — VERIFICATION RECORD (paste this whole block back to Muse)
import json, torch
record = {
    "model": "sdar:4b",
    "notebook_version": NOTEBOOK_VERSION,
    "ollada_version": ollada.__version__,
    "hardware": torch.cuda.get_device_name(0),
    "load_seconds": round(load_s, 1),
    "peak_vram_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2),
    "runs": results,
}
print(json.dumps(record, indent=2))